# BUSI 乳腺超声图像分类

本 notebook 用于：

1. 探索 BUSI 数据集（类别分布、样本图像）
2. 构建 ResNet 与 ViT 模型并查看参数量
3. 加载训练好的权重
4. 在测试集上评估并对比两个模型
5. 可视化混淆矩阵、预测样例与训练曲线


In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# 找到项目根目录（若在 notebooks/ 下运行则自动上溯一级）
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.py").exists() and (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from models import build_resnet, build_vit
from utils.dataset import BUSIDataset, build_loaders, get_file_paths
from utils.metrics import compute_metrics
from utils import visualization as viz

RESNET_NAME = f"resnet{config.RESNET_DEPTH}"

print("Project root:", PROJECT_ROOT)
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 1. 数据集探索

In [ ]:
samples = get_file_paths(config.DATA_DIR)
print("Total images (excluding masks):", len(samples))

counts = Counter(label for _, label in samples)
for c in config.CLASSES:
    print(f"  {c}: {counts.get(c, 0)}")

viz.plot_class_distribution(counts, config.CLASSES)
plt.show()


In [ ]:
from torchvision import transforms

# 仅用于展示的变换（不做归一化，方便可视化）
display_transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
])
display_ds = BUSIDataset(samples, config.CLASSES, transform=display_transform)
viz.plot_dataset_samples(display_ds, config.CLASSES, n_per_class=4)
plt.show()


## 2. 构建模型

In [ ]:
resnet = build_resnet(depth=config.RESNET_DEPTH, num_classes=config.NUM_CLASSES)
vit = build_vit(
    image_size=config.IMG_SIZE,
    patch_size=config.VIT_PATCH_SIZE,
    num_classes=config.NUM_CLASSES,
    dim=config.VIT_DIM,
    depth=config.VIT_DEPTH,
    heads=config.VIT_HEADS,
    mlp_dim=config.VIT_MLP_DIM,
    dropout=config.VIT_DROPOUT,
)

for name, model in [("ResNet", resnet), ("ViT", vit)]:
    n_params = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name}: {n_params/1e6:.2f}M params ({n_train/1e6:.2f}M trainable)")

dummy = torch.randn(2, 3, config.IMG_SIZE, config.IMG_SIZE)
with torch.no_grad():
    print("ResNet output shape:", tuple(resnet(dummy).shape))
    print("ViT output shape:", tuple(vit(dummy).shape))


## 3. 加载训练权重

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model(model, ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return model.to(device).eval()

resnet_ckpt = config.CHECKPOINT_DIR / f"{RESNET_NAME}_best.pt"
vit_ckpt = config.CHECKPOINT_DIR / "vit_best.pt"

resnet = load_model(resnet, resnet_ckpt) if resnet_ckpt.exists() else resnet.to(device)
vit = load_model(vit, vit_ckpt) if vit_ckpt.exists() else vit.to(device)

print("ResNet weights loaded:", resnet_ckpt.exists(), "->", resnet_ckpt)
print("ViT weights loaded:", vit_ckpt.exists(), "->", vit_ckpt)


## 4. 测试集评估与对比

In [ ]:
train_loader, val_loader, test_loader, classes = build_loaders(
    config.DATA_DIR,
    img_size=config.IMG_SIZE,
    batch_size=config.BATCH_SIZE,
    num_workers=0,  # notebook 中避免多进程问题
    ratios=(config.TRAIN_RATIO, config.VAL_RATIO, config.TEST_RATIO),
    seed=config.SEED,
)
print("Classes:", classes)
print("Train:", len(train_loader.dataset),
      "| Val:", len(val_loader.dataset),
      "| Test:", len(test_loader.dataset))


In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for images, labels in loader:
        images = images.to(device)
        logits = model(images)
        y_true.append(labels.numpy())
        y_pred.append(logits.argmax(dim=1).cpu().numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)

results = {}
for name, model in [("ResNet", resnet), ("ViT", vit)]:
    y_true, y_pred = predict(model, test_loader)
    results[name] = (y_true, y_pred, compute_metrics(y_true, y_pred))

rows = []
for name, (yt, yp, m) in results.items():
    rows.append({
        "Model": name,
        "Accuracy": m["accuracy"],
        "Precision(macro)": m["precision"],
        "Recall(macro)": m["recall"],
        "F1(macro)": m["f1"],
    })
summary = pd.DataFrame(rows).round(4)
summary


## 5. 结果可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, (yt, yp, m)) in zip(axes, results.items()):
    viz.plot_confusion_matrix(m["confusion_matrix"], classes,
                              title=f"{name} Confusion Matrix",
                              ax=ax, normalize=True)
plt.tight_layout()
plt.show()


In [ ]:
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.numpy()

with torch.no_grad():
    resnet_preds = resnet(images).argmax(dim=1).cpu().numpy()
    vit_preds = vit(images).argmax(dim=1).cpu().numpy()

n = min(8, len(images))
fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 3))
for i in range(n):
    img = viz.denormalize(images[i])
    axes[i].imshow(img.permute(1, 2, 0).numpy())
    axes[i].set_title(
        f"True: {classes[labels[i]]}\nResNet: {classes[resnet_preds[i]]}\nViT: {classes[vit_preds[i]]}",
        fontsize=8,
    )
    axes[i].axis("off")
plt.suptitle("Prediction comparison (input images)")
plt.tight_layout()
plt.show()


## 6. 训练曲线

In [ ]:
import json

def load_history(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

for name in [RESNET_NAME, "vit"]:
    hist_path = config.LOG_DIR / f"{name}_history.json"
    if hist_path.exists():
        viz.plot_training_curves(load_history(hist_path), title=f"{name} training curves")
        plt.show()
    else:
        print(f"No history found: {hist_path}")


## 7. 结论

- 两个模型都在 BUSI 三分类任务上训练，训练曲线保存在 `outputs/logs/`。
- 上方的表格与混淆矩阵对比了两个模型的 Accuracy / Precision / Recall / F1。
- 小数据集上，ResNet 依靠卷积的归纳偏置通常更容易收敛；ViT 往往需要更多数据
  或更强数据增强才能充分发挥。实际结论请以本地训练结果为准。
- 可在 `config.py` 与 `train.py` 的命令行参数中调整图像尺寸、ResNet 深度、
  ViT 的 patch size / dim / depth / heads、学习率、epoch 数等。
